In [277]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [278]:
customers = pd.read_csv("../data/processed/customers_clean.csv")

orders = pd.read_csv("../data/processed/orders_clean.csv")

order_items = pd.read_csv("../data/processed/order_items_clean.csv")

payments = pd.read_csv("../data/processed/payments_clean.csv")

reviews = pd.read_csv("../data/processed/reviews_clean.csv")

products = pd.read_csv("../data/processed/products_clean.csv")

sellers = pd.read_csv("../data/processed/sellers_clean.csv")

geolocation = pd.read_csv("../data/processed/geolocation_clean.csv")

category_translation = pd.read_csv(
    "../data/processed/category_translation_clean.csv"
)

In [279]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)
print("Category Translation:", category_translation.shape)

Customers: (99441, 5)
Orders: (99441, 12)
Order Items: (112650, 8)
Payments: (103886, 5)
Reviews: (99224, 8)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category Translation: (71, 2)


In [280]:
category_translation[
    ["product_category_name", "product_category_name_english"]
].head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [281]:
category_translation["product_category_name"].duplicated().sum()

np.int64(0)

In [282]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [283]:
# Merge Products with Category Translation

In [284]:
products = products.merge(
    category_translation,
    on = "product_category_name",
    how = "left"
)

In [285]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [286]:
products[
    [
        "product_id",
        "product_category_name",
        "product_category_name_english"
]].head(10)

,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,toys


In [287]:
# Check unmatched categories
products["product_category_name_english"].isnull().sum()

np.int64(623)

In [288]:
products[
    products["product_category_name"] == "unknown"
].shape

(610, 10)

In [289]:
products[
    products["product_category_name_english"].isnull()
]["product_category_name"].value_counts()

product_category_name
unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64

In [290]:
# 610 → products whose original category was missing. This is expected.
# 10 → real category exists, but translation is missing.
# 3 → another real category exists, but translation is missing.

In [291]:
# Handle the 610 expected unknowns
products["product_category_name_english"] = (
    products["product_category_name_english"].fillna("unknown")
)

In [292]:
products["product_category_name_english"].isnull().sum()

np.int64(0)

In [293]:
products[
    products["product_category_name"].isin([
        "portateis_cozinha_e_preparadores_de_alimentos",
        "pc_gamer"
    ])
][[
    "product_category_name",
    "product_category_name_english"
]].drop_duplicates()

,product_category_name,product_category_name_english
1628,pc_gamer,unknown
5821,portateis_cozinha_e_preparadores_de_alimentos,unknown


In [294]:
# Order Items + Products + Sellers

In [295]:
print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)

Order items: (112650, 8)
Products: (32951, 10)
Sellers: (3095, 4)


In [296]:
print("Missing product IDs in order items:",
      order_items["product_id"].isna().sum())

Missing product IDs in order items: 0


In [297]:
print("Missing seller IDs in order items:",
      order_items["seller_id"].isna().sum())

Missing seller IDs in order items: 0


In [298]:
# Merge Products

order_items_enriched = order_items.merge(
    products,
    on="product_id",
    how = "left",
    validate = "many_to_one"
)


In [299]:
order_items.columns.tolist()
#products.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'item_total']

In [300]:
products.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'product_category_name_english']

In [301]:
#Check the result

print("Before:",order_items.shape)
print("After:",order_items_enriched.shape)

Before: (112650, 8)
After: (112650, 17)


In [302]:
# Add Seller information

order_items_enriched = order_items_enriched.merge(
    sellers,
    on = "seller_id",
    how = "left",
    validate = "many_to_many"
)

In [303]:
order_items.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'item_total']

In [304]:
sellers.columns.tolist()

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

In [305]:
print("After seller merge:",
      order_items_enriched.shape)

After seller merge: (112650, 20)


In [306]:
# Inspect the enriched dataset
order_items_enriched[
    [
        "order_id",
        "product_id",
        "product_category_name_english",
        "seller_id",
        "price",
        "freight_value",
        "item_total",
        "seller_city",
        "seller_state"
    ]
].head()


,order_id,product_id,product_category_name_english,seller_id,price,freight_value,item_total,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,48436dade18ac8b2bce089ec2a041202,58.90,13.29,72.19,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93,259.83,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,5b51032eddd242adc84c38acab88f23d,199.00,17.87,216.87,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumery,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,25.78,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,garden_tools,df560393f3a51e74553ab94004ba5c87,199.90,18.14,218.04,loanda,PR


In [307]:
order_items_enriched.shape

(112650, 20)

In [308]:
# Create Order-Level Dataset



In [309]:
# Aggregate order items

order_items_summary = (
    order_items_enriched
    .groupby("order_id")
    .agg(
        total_items = ("order_item_id", "count"),
        total_product_price = ("price", "sum"),
        total_freight = ("freight_value", "sum"),
        total_item_amount = ("item_total", "sum"),
        unique_product = ("product_id", "nunique"),
        unique_sellers = ("seller_id", "nunique")
    )
    .reset_index()
)

In [310]:
order_items_summary

,order_id,total_items,total_product_price,total_freight,total_item_amount,unique_product,unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04,1,1
...,...,...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,1,299.99,43.41,343.40,1,1
98662,fffcd46ef2263f404302a634eb57f7eb,1,350.00,36.53,386.53,1,1
98663,fffce4705a9662cd70adb13d4a31832d,1,99.90,16.95,116.85,1,1
98664,fffe18544ffabc95dfada21779c9644f,1,55.99,8.72,64.71,1,1


In [311]:
order_items_summary.head()

,order_id,total_items,total_product_price,total_freight,total_item_amount,unique_product,unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04,1,1


In [312]:
print(order_items_summary.shape)

(98666, 7)


In [313]:
# order_item_summary.shape
# → (98666, 7)
# This means we reduced:
# 112,650 order-item records
#         ↓
#    group by order_id
#         ↓
# 98,666 unique orders

In [314]:
order_enriched = orders.merge(
    order_items_summary,
    on = "order_id",
    how = "left",
    validate="one_to_one"
)

In [315]:
order_enriched

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delivery_delay_days,delivery_performance,total_items,total_product_price,total_freight,total_item_amount,unique_product,unique_sellers
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,-7.107488,On Time / Early,1.0,29.99,8.72,38.71,1.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,-5.355729,On Time / Early,1.0,118.70,22.76,141.46,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,-17.245498,On Time / Early,1.0,159.90,19.22,179.12,1.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,-12.980069,On Time / Early,1.0,45.00,27.20,72.20,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,-9.238171,On Time / Early,1.0,19.90,8.72,28.62,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,8.218009,18.587442,-10.369433,On Time / Early,1.0,72.00,13.08,85.08,1.0,1.0
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,22.193727,23.459051,-1.265324,On Time / Early,1.0,174.90,20.10,195.00,1.0,1.0
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.859421,30.384225,-5.524803,On Time / Early,1.0,205.99,65.02,271.01,1.0,1.0
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.086424,37.105243,-20.018819,On Time / Early,2.0,359.98,81.18,441.16,1.0,1.0


In [316]:
print("Orders before:", orders.shape)
print("Orders after:", order_enriched.shape)

Orders before: (99441, 12)
Orders after: (99441, 18)


In [317]:
order_enriched.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'delivery_performance',
 'total_items',
 'total_product_price',
 'total_freight',
 'total_item_amount',
 'unique_product',
 'unique_sellers']

In [318]:
# Check for orders without item information

order_enriched[
    "total_item_amount"
].isna().sum()

np.int64(775)

In [319]:
order_enriched[
    order_enriched["total_item_amount"].isna()

]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [320]:
order_item_cols = [
    "total_items",
    "total_product_price",
    "total_freight",
    "total_item_amount",
    "unique_product",
    "unique_sellers"
]

order_enriched[order_item_cols] = (
    order_enriched[order_item_cols].fillna(0)
)

In [321]:
order_item_cols

['total_items',
 'total_product_price',
 'total_freight',
 'total_item_amount',
 'unique_product',
 'unique_sellers']

In [322]:
order_enriched[order_item_cols].isna().sum()

total_items            0
total_product_price    0
total_freight          0
total_item_amount      0
unique_product         0
unique_sellers         0
dtype: int64

In [323]:
# Connect Customers → Orders



In [324]:
# Check the relationship

print("Orders:", order_enriched.shape)
print("Customers:", customers.shape)

print(
    "Unique customer IDs in orders:",
    order_enriched["customer_id"].nunique()
)

print(
    "Unique customer IDs in customers:",
    customers["customer_id"].nunique()
)

Orders: (99441, 18)
Customers: (99441, 5)
Unique customer IDs in orders: 99441
Unique customer IDs in customers: 99441


In [325]:
# Merge
orders_customers = order_enriched.merge(
   customers,
   on = "customer_id",
   how = "left",
   validate = "many_to_one"
)


In [326]:
print("Before:", order_enriched.shape)
print("After:", orders_customers.shape)

Before: (99441, 18)
After: (99441, 22)


In [327]:
orders_customers[
    [
        "order_id",
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].head()

,order_id,customer_id,customer_unique_id,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP


In [328]:
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [329]:
orders_customers["customer_unique_id"].isna().sum()

np.int64(0)

In [330]:
print("Unique customers:",
      orders_customers["customer_unique_id"].nunique())

Unique customers: 96096


In [331]:
orders_per_customer = (
    orders_customers
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

In [332]:
orders_per_customer

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2    1
0000b849f77a49e4a4ce2b2a4ca5be3f    1
0000f46a3911fa3c0805444483337064    1
0000f6ccb0745a6a4b88665a16c9f078    1
0004aac84e0df4da2b147fca70cf8255    1
                                   ..
fffcf5a5ff07b0908bd4e2dbc735a684    1
fffea47cd6d3cc0a88bd621562a9d061    1
ffff371b4d645b6ecea244b27531430a    1
ffff5962728ec6157033ef9805bacc48    1
ffffd2657e2aad2907e67c3e9daecbeb    1
Name: order_id, Length: 96096, dtype: int64

In [333]:
orders_per_customer.describe()

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: order_id, dtype: float64

In [334]:
# Build Customer 360°


In [335]:
customer_360 = (
    orders_customers
    .groupby("customer_unique_id")
    .agg(
        total_orders = ("order_id", "nunique"),
        total_items = ("total_items", "sum"),
        total_spent = ("total_item_amount", "sum"),
        total_freight = ("total_freight", "sum"),
        unique_products = ("unique_product", "sum"),
        unique_sellers = ("unique_sellers", "sum"),
        avg_order_value = ("total_item_amount","mean")
    )
    .reset_index()
)


In [336]:
customer_360

,customer_unique_id,total_orders,total_items,total_spent,total_freight,unique_products,unique_sellers,avg_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1.0,141.90,12.00,1.0,1.0,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1.0,27.19,8.29,1.0,1.0,27.19
2,0000f46a3911fa3c0805444483337064,1,1.0,86.22,17.22,1.0,1.0,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,1.0,43.62,17.63,1.0,1.0,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,1.0,196.89,16.89,1.0,1.0,196.89
...,...,...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,1,2.0,2067.42,497.42,2.0,1.0,2067.42
96092,fffea47cd6d3cc0a88bd621562a9d061,1,1.0,84.58,19.69,1.0,1.0,84.58
96093,ffff371b4d645b6ecea244b27531430a,1,1.0,112.46,22.56,1.0,1.0,112.46
96094,ffff5962728ec6157033ef9805bacc48,1,1.0,133.69,18.69,1.0,1.0,133.69


In [337]:
customers.shape

(99441, 5)

In [338]:
customer_360.head()

,customer_unique_id,total_orders,total_items,total_spent,total_freight,unique_products,unique_sellers,avg_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1.0,141.90,12.00,1.0,1.0,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1.0,27.19,8.29,1.0,1.0,27.19
2,0000f46a3911fa3c0805444483337064,1,1.0,86.22,17.22,1.0,1.0,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,1.0,43.62,17.63,1.0,1.0,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,1.0,196.89,16.89,1.0,1.0,196.89


In [339]:
print(customer_360.shape)
print(customer_360.columns.tolist())

(96096, 8)
['customer_unique_id', 'total_orders', 'total_items', 'total_spent', 'total_freight', 'unique_products', 'unique_sellers', 'avg_order_value']


In [340]:
# adding Recency.

In [341]:
orders_customers["order_purchase_timestamp"] = pd.to_datetime(
    orders_customers["order_purchase_timestamp"]
)

In [342]:
# Find the latest purchase date in the dataset

max_purchase_date = orders_customers["order_purchase_timestamp"].max()

print("Latest purchase date:", max_purchase_date)


Latest purchase date: 2018-10-17 17:30:18


In [343]:
# customer's last purchase
customer_recency = (
    orders_customers
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .max()
    .reset_index()
)

In [344]:
customer_recency.columns = [
    "customer_unique_id",
    "last_purchase_date"
]

In [345]:
customer_recency.head()

,customer_unique_id,last_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42


In [346]:
customer_recency.nunique()

customer_unique_id    96096
last_purchase_date    95834
dtype: int64

In [347]:
# recency_days

customer_recency["recency_days"] =(
    max_purchase_date
    - customer_recency["last_purchase_date"]
).dt.days

In [348]:
customer_recency.head()

,customer_unique_id,last_purchase_date,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,160
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,163
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,585
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,369
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,336


In [349]:
customer_360 = customer_360.merge(
    customer_recency,
    on = "customer_unique_id",
    how = "left",
    validate = "one_to_one"
)

In [350]:
customer_360.shape

(96096, 10)

In [351]:
# Purchase Frequency


In [352]:
# Find first purchase date

customer_dates = (
    orders_customers
    .groupby("customer_unique_id")["order_purchase_timestamp"]
     .agg(
         first_purchase_date = "min",
         last_purchase_date  = "max"
     )
    .reset_index()

)

In [353]:
customer_dates["customer_lifetime_days"] = (
    customer_dates["last_purchase_date"]
    - customer_dates["first_purchase_date"]
).dt.days

In [354]:
customer_dates.head()

,customer_unique_id,first_purchase_date,last_purchase_date,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,0


In [355]:
# Calculate active months

customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days']

In [356]:
customer_dates.columns.tolist()

['customer_unique_id',
 'first_purchase_date',
 'last_purchase_date',
 'customer_lifetime_days']

In [357]:
customer_360 = customer_360.merge(
    customer_dates[
        [
           "customer_unique_id" ,
           "first_purchase_date",
           "last_purchase_date",
           "customer_lifetime_days"
        ]
    ],
    on = "customer_unique_id",
    how = "left",
    validate = "one_to_one"
)

In [358]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date_x',
 'recency_days',
 'first_purchase_date',
 'last_purchase_date_y',
 'customer_lifetime_days']

In [359]:
customer_360.shape

(96096, 13)

In [360]:
customer_360["active_months"] = (
    customer_360["customer_lifetime_days"] /30
)

In [361]:
customer_360["active_months"]

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96091    0.0
96092    0.0
96093    0.0
96094    0.0
96095    0.0
Name: active_months, Length: 96096, dtype: float64

In [362]:
customer_360[
    [
        "customer_unique_id",
        "customer_lifetime_days",
        "active_months"
    ]
].head()

,customer_unique_id,customer_lifetime_days,active_months
0,0000366f3b9a7992bf8c76cfdf3221e2,0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,0,0.0
2,0000f46a3911fa3c0805444483337064,0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,0,0.0
4,0004aac84e0df4da2b147fca70cf8255,0,0.0


In [363]:
customer_360.shape

(96096, 14)

In [364]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date_x',
 'recency_days',
 'first_purchase_date',
 'last_purchase_date_y',
 'customer_lifetime_days',
 'active_months']

In [365]:
# Review Behavior

In [366]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,Positive
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,Positive
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,Positive
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,Positive
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,Positive


In [367]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
review_sentiment               0
dtype: int64

In [368]:
customer_reviews = (
    orders_customers
    .merge(
        reviews[["order_id", "review_score"]],
        on = "order_id",
        how = "left"
    )
     .groupby("customer_unique_id")
     .agg(
        avg_review_score = ("review_score","mean"),
        total_reviews = ("review_score", "count")
    ).reset_index()
)

In [369]:
customer_reviews.head()

,customer_unique_id,avg_review_score,total_reviews
0,0000366f3b9a7992bf8c76cfdf3221e2,5.0,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4.0,1
2,0000f46a3911fa3c0805444483337064,3.0,1
3,0000f6ccb0745a6a4b88665a16c9f078,4.0,1
4,0004aac84e0df4da2b147fca70cf8255,5.0,1


In [370]:
customer_reviews.shape

(96096, 3)

In [371]:
orders_customers.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,...,total_items,total_product_price,total_freight,total_item_amount,unique_product,unique_sellers,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,...,1.0,29.99,8.72,38.71,1.0,1.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,...,1.0,118.70,22.76,141.46,1.0,1.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,...,1.0,159.90,19.22,179.12,1.0,1.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,...,1.0,45.00,27.20,72.20,1.0,1.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,...,1.0,19.90,8.72,28.62,1.0,1.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [372]:
# Check missing reviews

customer_reviews["avg_review_score"].isna().sum()

np.int64(716)

In [373]:
customer_360 = customer_360.merge(
    customer_reviews,
    on = "customer_unique_id",
    how = "left",
    validate = "one_to_one"
)

In [374]:
customer_360.shape

(96096, 16)

In [375]:
# check how many customers have no review

customer_360["avg_review_score"].isna().sum()



np.int64(716)

In [376]:
customer_360["total_reviews"].isna().sum()

np.int64(0)

In [377]:
customer_360["has_review"] = (
    customer_360["total_reviews"] > 0
).astype(int)

In [378]:
customer_360["has_review"].value_counts()

has_review
1    95380
0      716
Name: count, dtype: int64

In [379]:
# Payment Behavior

In [380]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8.0,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1.0,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1.0,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8.0,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2.0,128.45


In [381]:
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    2
payment_value           0
dtype: int64

In [382]:
#create payment summary per order

payment_order_summary = (
    payments
    .groupby("order_id")
    .agg(
        order_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

In [383]:
type(payment_order_summary)

pandas.DataFrame

In [384]:
payment_order_summary.shape

(99440, 4)

In [385]:
payments.shape


(103886, 5)

In [386]:
orders_customers = orders_customers.merge(
    payment_order_summary,
    on = "order_id",
    how = "left",
    validate = "one_to_one"
)

In [387]:
orders_customers.shape

(99441, 25)

In [388]:
# missing payment record
orders_customers[
    orders_customers["order_payment_value"].isna()
][
    ["order_id", "order_status"]
]


,order_id,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,delivered


In [389]:
orders_customers[
    orders_customers["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"
][[
    "order_id",
    "order_status",
    "order_payment_value",
    "payment_count",
    "max_installments"
]]


,order_id,order_status,order_payment_value,payment_count,max_installments
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,NaN,NaN,NaN


In [390]:
orders_customers.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'delivery_performance',
 'total_items',
 'total_product_price',
 'total_freight',
 'total_item_amount',
 'unique_product',
 'unique_sellers',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_payment_value',
 'payment_count',
 'max_installments']

In [391]:
# Customer Payment Features

In [392]:
customer_payment = (
    orders_customers
    .groupby("customer_unique_id")
    .agg(
        total_payment = ("order_payment_value", "sum"),
        avg_payment_value = ("order_payment_value","mean"),
        total_payment_records = ("payment_count","sum"),
        max_installments = ("max_installments", "max")
    )
    .reset_index()
)

In [393]:
customer_payment.shape

(96096, 5)

In [394]:
customer_payment.head()

,customer_unique_id,total_payment,avg_payment_value,total_payment_records,max_installments
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,141.90,1.0,8.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,27.19,1.0,1.0
2,0000f46a3911fa3c0805444483337064,86.22,86.22,1.0,8.0
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,43.62,1.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,196.89,196.89,1.0,6.0


In [395]:
customer_360 = customer_360.merge(
    customer_payment,
    on = "customer_unique_id",
    how = "left",
    validate = "one_to_one"
)

In [396]:
# Check the new shape

customer_360.shape

(96096, 21)

In [397]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date_x',
 'recency_days',
 'first_purchase_date',
 'last_purchase_date_y',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments']

In [398]:
(
    customer_360["last_purchase_date_x"]
    == customer_360["last_purchase_date_y"]
).all()

np.True_

In [399]:
customer_360 = customer_360.drop(
    columns = ["last_purchase_date_y"]
)

In [400]:
customer_360.shape

(96096, 20)

In [401]:
customer_360 = customer_360.rename(
    columns = {"last_purchase_date_x": "last_purchase_date"}
)

In [402]:
customer_360.shape

(96096, 20)

In [403]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments']

In [404]:
customer_360[
    [
        "total_payment",
        "avg_payment_value",
        "total_payment_records",
        "max_installments"
    ]
].isna().sum()

total_payment            0
avg_payment_value        1
total_payment_records    0
max_installments         3
dtype: int64

In [405]:
customer_360[
    customer_360["avg_payment_value"].isna() |
    customer_360["max_installments"].isna()
][[
    "customer_unique_id",
    "total_orders",
    "total_payment",
    "avg_payment_value",
    "total_payment_records",
    "max_installments"
]]

,customer_unique_id,total_orders,total_payment,avg_payment_value,total_payment_records,max_installments
49312,830d5b7aaa3b6f1e9ad63703bec97d23,1,0.00,NaN,0.0,NaN
57518,9925e1d7dff0d807355599dee04830ab,1,129.94,129.94,1.0,NaN
92130,f54cea27c80dc09bfe07b1cf1e01b845,1,58.69,58.69,1.0,NaN


In [406]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [407]:
# Create payment-type flags

payments_type = payments.copy()

In [408]:
payments_type["credit_card_flag"] = (
    payments_type["payment_type"] == "credit_card"
).astype(int)

payments_type["boleto_flag"] = (
    payments_type["payment_type"] == "boleto"
).astype(int)

payments_type["voucher_flag"] =(
    payments_type["payment_type"] == "voucher"
).astype(int)


payments_type["debit_card_flag"] =(
    payments_type["payment_type"]  =="debit_card"
).astype(int)



In [409]:
payments_type.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value,credit_card_flag,boleto_flag,voucher_flag,debit_card_flag
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8.0,99.33,1,0,0,0
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1.0,24.39,1,0,0,0
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1.0,65.71,1,0,0,0
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8.0,107.78,1,0,0,0
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2.0,128.45,1,0,0,0


In [410]:
payments_customer = payments_type.merge(
    orders[["order_id", "customer_id"]],
    on = "order_id",
    how = "left",
    validate = "many_to_one"
)

In [411]:
payments_customer

,order_id,payment_sequential,payment_type,payment_installments,payment_value,credit_card_flag,boleto_flag,voucher_flag,debit_card_flag,customer_id
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8.0,99.33,1,0,0,0,0a8556ac6be836b46b3e89920d59291c
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1.0,24.39,1,0,0,0,f2c7fc58a9de810828715166c672f10a
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1.0,65.71,1,0,0,0,25b14b69de0b6e184ae6fe2755e478f9
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8.0,107.78,1,0,0,0,7a5d8efaaa1081f800628c30d2b0728f
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2.0,128.45,1,0,0,0,15fd6fb8f8312dbb4674e4518d6fa3b3
...,...,...,...,...,...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1.0,363.31,0,1,0,0,5d576cb2dfa3bc05612c392a1ee9c654
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2.0,96.80,1,0,0,0,2079230c765a88530822a34a4cec2aa0
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1.0,47.77,1,0,0,0,e4abb5057ec8cfda9759c0dc415a8188
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5.0,369.54,1,0,0,0,5d719b0d300663188169c6560e243f27


In [412]:
payments_customer = payments_customer.merge(
    customers[["customer_id","customer_unique_id"]],
    on = "customer_id",
    how = "left",
    validate = "many_to_one"
)

In [413]:
customer_payment_types = (
    payments_customer
    .groupby("customer_unique_id")
    .agg(
        credit_card_payments = ("credit_card_flag", "sum"),
        boleto_payments = ("boleto_flag", "sum"),
        voucher_payments = ("voucher_flag", "sum"),
        debit_card_payments = ("debit_card_flag", "sum")
    )
    .reset_index()
)

In [414]:
customer_payment_types.shape

(96095, 5)

In [415]:
customer_payment_types.head()

,customer_unique_id,credit_card_payments,boleto_payments,voucher_payments,debit_card_payments
0,0000366f3b9a7992bf8c76cfdf3221e2,1,0,0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,0,0,0
2,0000f46a3911fa3c0805444483337064,1,0,0,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,0,0,0
4,0004aac84e0df4da2b147fca70cf8255,1,0,0,0


In [416]:
customer_360 = customer_360.merge(
    customer_payment_types,
    on = "customer_unique_id",
    how = "left",
    validate = "one_to_one"
)

In [417]:
customer_360.shape

(96096, 24)

In [418]:
customer_payment_types[
    [
        "credit_card_payments",
        "boleto_payments",
        "voucher_payments",
        "debit_card_payments"
    ]
].head()

,credit_card_payments,boleto_payments,voucher_payments,debit_card_payments
0,1,0,0,0
1,1,0,0,0
2,1,0,0,0
3,1,0,0,0
4,1,0,0,0


In [419]:
customer_360[
    [
        "credit_card_payments",
        "boleto_payments",
        "voucher_payments",
        "debit_card_payments"
    ]
].isna().sum()

credit_card_payments    1
boleto_payments         1
voucher_payments        1
debit_card_payments     1
dtype: int64

In [420]:
payment_type_cols = [
    "credit_card_payments",
    "boleto_payments",
    "voucher_payments",
    "debit_card_payments"
]

customer_360[payment_type_cols] = (
    customer_360[payment_type_cols].fillna(0)
)

In [421]:
customer_360[payment_type_cols].isna().sum()

credit_card_payments    0
boleto_payments         0
voucher_payments        0
debit_card_payments     0
dtype: int64

In [422]:
customer_360.shape

(96096, 24)

In [423]:
#Check delivery columns

[c for c in orders_customers.columns if "delivery" in c.lower()]

['order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'delivery_performance']

In [424]:
orders_customers.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'delivery_performance',
 'total_items',
 'total_product_price',
 'total_freight',
 'total_item_amount',
 'unique_product',
 'unique_sellers',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_payment_value',
 'payment_count',
 'max_installments']

In [425]:
customer_delivery = (
    orders_customers
    .groupby("customer_unique_id")
    .agg(
        avg_delivery_days=("delivery_days", "mean"),
        max_delivery_days=("delivery_days", "max"),
        avg_delivery_delay=("delivery_delay_days", "mean"),
        late_orders=("delivery_performance", lambda x: (x == "Late").sum()),
        on_time_orders=("delivery_performance", lambda x: (x == "On Time/Early").sum())
    )
    .reset_index()
)


In [426]:
customer_delivery.head()

,customer_unique_id,avg_delivery_days,max_delivery_days,avg_delivery_delay,late_orders,on_time_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,6.411227,6.411227,-4.132905,0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3.285590,3.285590,-4.248125,0,0
2,0000f46a3911fa3c0805444483337064,25.731759,25.731759,-1.389734,0,0
3,0000f6ccb0745a6a4b88665a16c9f078,20.037083,20.037083,-11.108970,0,0
4,0004aac84e0df4da2b147fca70cf8255,13.141134,13.141134,-7.035463,0,0


In [427]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments']

In [428]:
customer_360.shape

(96096, 24)

In [429]:
# customer_360[
#     [
#         "avg_delivery_days",
#         "max_delivery_days",
#         "avg_delivery_delay",
#         "late_orders",
#         "on_time_orders"
#     ]
    
# ].isna().sum()

In [430]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments']

In [431]:
customer_delivery.columns.tolist()

['customer_unique_id',
 'avg_delivery_days',
 'max_delivery_days',
 'avg_delivery_delay',
 'late_orders',
 'on_time_orders']

In [432]:
customer_360.columns

Index(['customer_unique_id', 'total_orders', 'total_items', 'total_spent',
       'total_freight', 'unique_products', 'unique_sellers', 'avg_order_value',
       'last_purchase_date', 'recency_days', 'first_purchase_date',
       'customer_lifetime_days', 'active_months', 'avg_review_score',
       'total_reviews', 'has_review', 'total_payment', 'avg_payment_value',
       'total_payment_records', 'max_installments', 'credit_card_payments',
       'boleto_payments', 'voucher_payments', 'debit_card_payments'],
      dtype='str')

In [433]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments']

In [434]:
[c for c in customer_delivery.columns
 if c != "customer_unique_id" and c in customer_360.columns]

[]

In [435]:
customer_360 = customer_360.merge(
    customer_delivery,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

In [436]:
customer_360.shape

(96096, 29)

In [437]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments',
 'avg_delivery_days',
 'max_delivery_days',
 'avg_delivery_delay',
 'late_orders',
 'on_time_orders']

In [438]:
[c for c in customer_360.columns if "_x" in c or "_y" in c]

[]

In [439]:
customer_360.filter(
    regex="_x$|_y$"
).head()

""
0
1
2
3
4


In [440]:
customer_360 = customer_360.drop(
    columns=[
        "total_payment_x",
        "avg_payment_value_x",
        "total_payment_records_x",
        "max_installments_x",
        "total_payment_y",
        "avg_payment_value_y",
        "total_payment_records_y",
        "max_installments_y"
    ]
)

KeyError: "['total_payment_x', 'avg_payment_value_x', 'total_payment_records_x', 'max_installments_x', 'total_payment_y', 'avg_payment_value_y', 'total_payment_records_y', 'max_installments_y'] not found in axis"

In [ ]:
customer_360.shape

(96096, 29)

In [ ]:
[c for c in customer_360.columns if "payment" in c]

['total_payment',
 'avg_payment_value',
 'total_payment_records',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments']

In [ ]:
customer_360.shape

(96096, 29)

In [ ]:
customer_360[
    [
        "avg_delivery_days",
        "max_delivery_days",
        "avg_delivery_delay",
        "late_orders",
        "on_time_orders"
    ]
].isna().sum()

avg_delivery_days     2740
max_delivery_days     2740
avg_delivery_delay    2740
late_orders              0
on_time_orders           0
dtype: int64

In [ ]:
orders_customers[
    orders_customers["delivery_days"].isna()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [ ]:
# create late_order_rate

customer_360["late_order_rate"] = (
    customer_360["late_orders"] /
    customer_360["total_orders"]
)

In [ ]:
customer_360["late_order_rate"]

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96091    0.0
96092    0.0
96093    0.0
96094    0.0
96095    0.0
Name: late_order_rate, Length: 96096, dtype: float64

In [ ]:
customer_360[
    [
        "customer_unique_id",
        "total_orders",
        "late_orders",
        "on_time_orders",
        "late_order_rate"
    ]
].head()

,customer_unique_id,total_orders,late_orders,on_time_orders,late_order_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,1,0,0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,0,0,0.0
2,0000f46a3911fa3c0805444483337064,1,0,0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,0,0,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,0,0,0.0


In [ ]:
customer_360["late_order_rate"].describe()

count    96096.000000
mean         0.079107
std          0.268357
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: late_order_rate, dtype: float64

In [ ]:
customer_360["on_time_order_rate"] = (
    customer_360["on_time_orders"] / 
    customer_360["total_orders"]
)

In [ ]:
customer_360[
    [
        "customer_unique_id",
        "total_orders",
        "late_orders",
        "on_time_orders",
        "late_order_rate",
        "on_time_order_rate"
    ]
].head()

,customer_unique_id,total_orders,late_orders,on_time_orders,late_order_rate,on_time_order_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,1,0,0,0.0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,0,0,0.0,0.0
2,0000f46a3911fa3c0805444483337064,1,0,0,0.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,0,0,0.0,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,0,0,0.0,0.0


In [ ]:
customer_360["on_time_order_rate"].describe()

count    96096.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: on_time_order_rate, dtype: float64

In [ ]:
orders_customers["delivery_performance"].value_counts(dropna=False)

delivery_performance
On Time / Early    88649
Late                7827
Not Delivered       2965
Name: count, dtype: int64

In [ ]:
customer_360 = customer_360.drop(
    columns = ["on_time_order_rate"]
)

In [ ]:
delivery_lookup = customer_delivery.set_index("customer_unique_id")

customer_360["on_time_orders"] = (
    customer_360["customer_unique_id"]
    .map(delivery_lookup["on_time_orders"])
)

In [ ]:
customer_360["on_time_order_rate"] = (
    customer_360["on_time_orders"] /
    customer_360["total_orders"]
)

In [ ]:
customer_360["on_time_order_rate"].describe()

count    96096.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: on_time_order_rate, dtype: float64

In [ ]:
customer_delivery["on_time_orders"].value_counts()

on_time_orders
0    96096
Name: count, dtype: int64

In [ ]:
orders_customers["delivery_performance"].unique()

<ArrowStringArray>
['On Time / Early', 'Not Delivered', 'Late']
Length: 3, dtype: str

In [ ]:
on_time_summary = (
    orders_customers
    .assign(
        on_time_flag=(
            orders_customers["delivery_performance"]
            == "On Time / Early"
        ).astype(int)
    )
    .groupby("customer_unique_id")["on_time_flag"]
    .sum()
    .reset_index(name="on_time_orders")
)

In [ ]:
on_time_summary["on_time_orders"].value_counts()

on_time_orders
1     83422
0     10200
2      2273
3       159
4        27
5         7
6         4
7         2
8         1
15        1
Name: count, dtype: int64

In [ ]:
customer_360["on_time_order_rate"] = (
    customer_360["on_time_orders"] /
    customer_360["total_orders"]
)

In [ ]:
customer_360["on_time_order_rate"].describe()

count    96096.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: on_time_order_rate, dtype: float64

In [ ]:
customer_360["customer_unique_id"].head()

0    0000366f3b9a7992bf8c76cfdf3221e2
1    0000b849f77a49e4a4ce2b2a4ca5be3f
2    0000f46a3911fa3c0805444483337064
3    0000f6ccb0745a6a4b88665a16c9f078
4    0004aac84e0df4da2b147fca70cf8255
Name: customer_unique_id, dtype: str

In [ ]:
on_time_summary["customer_unique_id"].head()

0    0000366f3b9a7992bf8c76cfdf3221e2
1    0000b849f77a49e4a4ce2b2a4ca5be3f
2    0000f46a3911fa3c0805444483337064
3    0000f6ccb0745a6a4b88665a16c9f078
4    0004aac84e0df4da2b147fca70cf8255
Name: customer_unique_id, dtype: str

In [ ]:
customer_360 = customer_360.drop(
    columns=["on_time_orders", "on_time_order_rate"]
)

In [ ]:
customer_360 = customer_360.merge(
    on_time_summary,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
customer_360["on_time_order_rate"] = (
    customer_360["on_time_orders"]
    / customer_360["total_orders"]
)

In [ ]:
customer_360["on_time_order_rate"].describe()

count    96096.000000
mean         0.891378
std          0.309150
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: on_time_order_rate, dtype: float64

In [ ]:
customer_360[
    [
        "total_orders",
        "on_time_orders",
        "on_time_order_rate"
    ]
].head(10)

,total_orders,on_time_orders,on_time_order_rate
0,1,1,1.0
1,1,1,1.0
2,1,1,1.0
3,1,1,1.0
4,1,1,1.0
5,1,1,1.0
6,1,1,1.0
7,1,1,1.0
8,1,1,1.0
9,1,0,0.0


In [ ]:
not_delivered_summary = (
    orders_customers
    .groupby("customer_unique_id")["delivery_performance"]
     .apply(lambda x : (x == "Not Delivered").sum()) 
      .reset_index(name = "not delivered_orders")
)

In [ ]:
not_delivered_summary["not delivered_orders"].value_counts()

not delivered_orders
0    93157
1     2915
2       22
3        2
Name: count, dtype: int64

In [ ]:
customer_360 = customer_360.merge(
    not_delivered_summary,
    on = "customer_unique_id",
    how = "left",
    validate = "one_to_one"
)

In [ ]:
customer_360.shape

(96096, 32)

In [ ]:
customer_360 = customer_360.rename(
    columns = {
        "not delivered_orders": "not_delivered_orders"
    }
)

In [ ]:
customer_360[
    [
        "customer_unique_id",
        "total_orders",
        "on_time_orders",
        "late_orders",
        "not_delivered_orders",
        "on_time_order_rate",
        "late_order_rate"
    ]
].head(10)

,customer_unique_id,total_orders,on_time_orders,late_orders,not_delivered_orders,on_time_order_rate,late_order_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,0,0,1.0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,0,0,1.0,0.0
2,0000f46a3911fa3c0805444483337064,1,1,0,0,1.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,0,0,1.0,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,1,0,0,1.0,0.0
5,0004bd2a26a76fe21f786e4fbd80607f,1,1,0,0,1.0,0.0
6,00050ab1314c0e55a6ca13cf7181fecf,1,1,0,0,1.0,0.0
7,00053a61a98854899e70ed204dd4bafe,1,1,0,0,1.0,0.0
8,0005e1862207bf6ccc02e4228effd9a0,1,1,0,0,1.0,0.0
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1,0,1,0,0.0,1.0


In [ ]:
customer_360["not_delivered_orders"].isna().sum()

np.int64(0)

In [ ]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments',
 'avg_delivery_days',
 'max_delivery_days',
 'avg_delivery_delay',
 'late_orders',
 'late_order_rate',
 'on_time_orders',
 'on_time_order_rate',
 'not_delivered_orders']

In [ ]:
customer_360 = customer_360.rename(
    columns = {
        "total_spent": "total_customer_spent",
        "avg_order_value": "average_order_value",
        "late_orders": "late_order_count"
    }
)

In [ ]:
customer_360["customer_unique_id"].duplicated().sum()

np.int64(0)

In [ ]:
customer_360.isna().sum().sort_values(ascending=False).head(15)

avg_delivery_delay     2740
avg_delivery_days      2740
max_delivery_days      2740
avg_review_score        716
max_installments          3
avg_payment_value         1
total_items               0
customer_unique_id        0
average_order_value       0
last_purchase_date        0
first_purchase_date       0
total_orders              0
total_freight             0
unique_products           0
unique_sellers            0
dtype: int64

In [ ]:
customer_360[
    customer_360["max_installments"].isna() |
    customer_360["avg_payment_value"].isna()
][
    [
        "customer_unique_id",
        "total_orders",
        "total_payment",
        "avg_payment_value",
        "total_payment_records",
        "max_installments"
    ]
]

,customer_unique_id,total_orders,total_payment,avg_payment_value,total_payment_records,max_installments
49312,830d5b7aaa3b6f1e9ad63703bec97d23,1,0.00,NaN,0.0,NaN
57518,9925e1d7dff0d807355599dee04830ab,1,129.94,129.94,1.0,NaN
92130,f54cea27c80dc09bfe07b1cf1e01b845,1,58.69,58.69,1.0,NaN


In [ ]:
payments[
    payments["order_id"].isin(
        orders_customers[
            orders_customers["customer_unique_id"].isin(
                customer_360[
                    customer_360["max_installments"].isna() |
                    customer_360["avg_payment_value"].isna()
                ]["customer_unique_id"]
            )
        ]["order_id"]
    )
][
    [
        "order_id",
        "payment_type",
        "payment_value",
        "payment_installments"
    ]
]

,order_id,payment_type,payment_value,payment_installments
46982,744bade1fcf9ff3f31d860ace076d422,credit_card,58.69,NaN
79014,1a57108394169c0b47d8f876acc9ba2d,credit_card,129.94,NaN


In [ ]:
customer_360.shape

(96096, 32)

In [ ]:
customer_360["customer_unique_id"].duplicated().sum()

np.int64(0)

In [ ]:
customer_360.isna().sum().sort_values(ascending=False).head(15)

avg_delivery_delay     2740
avg_delivery_days      2740
max_delivery_days      2740
avg_review_score        716
max_installments          3
avg_payment_value         1
total_items               0
customer_unique_id        0
average_order_value       0
last_purchase_date        0
first_purchase_date       0
total_orders              0
total_freight             0
unique_products           0
unique_sellers            0
dtype: int64

In [ ]:
[c for c in customer_360.columns if "order_value" in c]

['average_order_value']

In [ ]:
customer_360.shape

(96096, 32)

In [ ]:
customer_360["customer_unique_id"].duplicated().sum()


np.int64(0)

In [ ]:
customer_360.duplicated().sum()

np.int64(0)

In [ ]:
customer_360[
    [
        "total_orders",
        "total_items",
        "total_customer_spent",
        "total_freight",
        "average_order_value",
        "total_payment",
        "avg_payment_value",
        "avg_delivery_days",
        "avg_delivery_delay"
    ]
].describe()

,total_orders,total_items,total_customer_spent,total_freight,average_order_value,total_payment,avg_payment_value,avg_delivery_days,avg_delivery_delay
count,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96095.000000,93356.000000,93356.000000
mean,1.034809,1.172265,164.872141,23.433957,159.810187,166.592492,161.401796,12.567974,-11.151587
std,0.214384,0.627071,227.938658,22.883215,220.564409,231.428332,222.308078,9.546557,10.142746
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.533414,-146.016123
25%,1.000000,1.000000,62.390000,13.920000,61.690000,63.120000,62.460000,6.788322,-16.227451
50%,1.000000,1.000000,107.270000,17.530000,105.180000,108.000000,105.830000,10.234311,-11.748356
75%,1.000000,1.000000,182.237500,25.420000,176.230000,183.530000,177.210000,15.718643,-6.392506
max,17.000000,24.000000,13664.080000,1794.960000,13664.080000,13664.080000,13664.080000,209.628611,188.975081


In [ ]:
customer_360.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   total_orders            96096 non-null  int64         
 2   total_items             96096 non-null  float64       
 3   total_customer_spent    96096 non-null  float64       
 4   total_freight           96096 non-null  float64       
 5   unique_products         96096 non-null  float64       
 6   unique_sellers          96096 non-null  float64       
 7   average_order_value     96096 non-null  float64       
 8   last_purchase_date      96096 non-null  datetime64[us]
 9   recency_days            96096 non-null  int64         
 10  first_purchase_date     96096 non-null  datetime64[us]
 11  customer_lifetime_days  96096 non-null  int64         
 12  active_months           96096 non-null  float64       
 1

In [ ]:
customer_360.to_csv(
    "../data/processed/customer_360.csv",
    index=False
)

In [ ]:
import os

os.path.exists("../data/processed/customer_360.csv")

True

In [ ]:
customer_360.to_csv(
    "../data/processed/customer_360.csv",
    index=False
)

In [ ]:
import os

os.path.exists("../data/processed/customer_360.csv")

True

In [ ]:
customer_360.shape

(96096, 29)

In [441]:
customer_360.shape

(96096, 29)

In [442]:
customer_360.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_items',
 'total_spent',
 'total_freight',
 'unique_products',
 'unique_sellers',
 'avg_order_value',
 'last_purchase_date',
 'recency_days',
 'first_purchase_date',
 'customer_lifetime_days',
 'active_months',
 'avg_review_score',
 'total_reviews',
 'has_review',
 'total_payment',
 'avg_payment_value',
 'total_payment_records',
 'max_installments',
 'credit_card_payments',
 'boleto_payments',
 'voucher_payments',
 'debit_card_payments',
 'avg_delivery_days',
 'max_delivery_days',
 'avg_delivery_delay',
 'late_orders',
 'on_time_orders']